In [1]:
import pandas as pd

In [ ]:
df = pd.read_csv("../order_items.csv", dtype={"promo_id": "string", "promo_id_2": "string"})
df.head()

,order_id,product_id,quantity,unit_price,discount_amount,promo_id,promo_id_2
0,1,2400,7,1138.22,0.0,<NA>,<NA>
1,2,609,7,10166.25,0.0,<NA>,<NA>
2,3,396,3,11220.33,0.0,<NA>,<NA>
3,4,635,5,10639.25,0.0,<NA>,<NA>
4,6,1935,1,1597.84,0.0,<NA>,<NA>


In [3]:
# Xem kiểu dữ liệu và có giá trị null không
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 714669 entries, 0 to 714668
Data columns (total 7 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   order_id         714669 non-null  int64  
 1   product_id       714669 non-null  int64  
 2   quantity         714669 non-null  int64  
 3   unit_price       714669 non-null  float64
 4   discount_amount  714669 non-null  float64
 5   promo_id         276316 non-null  string 
 6   promo_id_2       206 non-null     string 
dtypes: float64(2), int64(3), string(2)
memory usage: 38.2 MB


In [4]:
# Thống kê mô tả các cột số
df.describe()

,order_id,product_id,quantity,unit_price,discount_amount
count,714669.000000,714669.000000,714669.000000,714669.000000,714669.000000
mean,411615.076561,1234.931370,4.495988,5114.690157,1048.887415
std,240480.310686,691.332564,2.290143,3774.817912,2280.530606
min,1.000000,1.000000,1.000000,392.570000,0.000000
25%,203229.000000,689.000000,2.000000,1906.890000,0.000000
50%,409306.000000,990.000000,4.000000,4257.770000,0.000000
75%,618981.000000,2045.000000,6.000000,7273.760000,967.630000
max,834397.000000,2412.000000,8.000000,43056.000000,35235.470000


In [5]:
# Kiểm tra số lượng null trong các cột
df.isna().sum()

order_id                0
product_id              0
quantity                0
unit_price              0
discount_amount         0
promo_id           438353
promo_id_2         714463
dtype: int64

In [6]:
# Xem phân bố giá trị của promo_id và promo_id_2
# Lưu ý: giá trị null ở hai cột này là hợp lệ (đơn hàng không áp dụng mã giảm giá)
display(df["promo_id"].value_counts(dropna=False).head(10))
display(df["promo_id_2"].value_counts(dropna=False))

promo_id
<NA>          438353
PROMO-0014     11451
PROMO-0010     11345
PROMO-0004     11126
PROMO-0020     10121
PROMO-0011      9594
PROMO-0007      9373
PROMO-0021      8966
PROMO-0017      8808
PROMO-0001      8523
Name: count, dtype: Int64

promo_id_2
<NA>          714463
PROMO-0015       132
PROMO-0025        74
Name: count, dtype: Int64

In [7]:
# Kiểm tra trùng lặp toàn bộ dòng
print("Số dòng trùng toàn bộ dòng:", df.duplicated().sum())

# Kiểm tra trùng cặp khóa (order_id, product_id)
pk_cols = ["order_id", "product_id"]
dup_pk = df.duplicated(subset=pk_cols, keep=False)
print(f"Số dòng trùng cặp (order_id, product_id): {dup_pk.sum()}")

# Hiển thị các dòng trùng để kiểm tra
display(df[dup_pk].sort_values(pk_cols).head(10))

Số dòng trùng toàn bộ dòng: 0
Số dòng trùng cặp (order_id, product_id): 32


,order_id,product_id,quantity,unit_price,discount_amount,promo_id,promo_id_2
12233,14280,976,1,4019.47,0.00,<NA>,<NA>
12234,14280,976,2,3937.99,0.00,<NA>,<NA>
99645,113379,786,6,694.34,0.00,<NA>,<NA>
99646,113379,786,1,699.37,0.00,<NA>,<NA>
189239,215525,1859,5,1896.11,0.00,<NA>,<NA>
189240,215525,1859,5,1897.20,0.00,<NA>,<NA>
190291,216740,791,8,793.10,0.00,<NA>,<NA>
190292,216740,791,5,825.52,0.00,<NA>,<NA>
214311,243342,777,7,1181.17,1653.64,PROMO-0010,<NA>
214312,243342,777,5,1147.21,1147.21,PROMO-0010,<NA>


In [8]:
# Vì cùng order_id và product_id nhưng số lượng hoặc giá khác nhau (khách mua nhiều lần sản phẩm đó trong 1 đơn),
# nên ta giữ lại toàn bộ dữ liệu và thêm surrogate key item_id để định danh duy nhất từng dòng sản phẩm
df["item_id"] = range(1, len(df) + 1)
print("item_id is unique:", df["item_id"].is_unique)

item_id is unique: True


In [9]:
# Kiểm tra các giá trị dị thường (số âm)
print("quantity <= 0       :", (df["quantity"] <= 0).sum())
print("unit_price <= 0     :", (df["unit_price"] <= 0).sum())
print("discount_amount < 0 :", (df["discount_amount"] < 0).sum())

quantity <= 0       : 0
unit_price <= 0     : 0
discount_amount < 0 : 0


In [10]:
# Ép kiểu dữ liệu chuẩn (Schema mapping)
df["item_id"] = df["item_id"].astype("int32")
df["order_id"] = df["order_id"].astype("int32")
df["product_id"] = df["product_id"].astype("int16")
df["quantity"] = df["quantity"].astype("int8")
df["unit_price"] = df["unit_price"].astype("float64")
df["discount_amount"] = df["discount_amount"].astype("float64")
df["promo_id"] = df["promo_id"].astype("category")
df["promo_id_2"] = df["promo_id_2"].astype("category")

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 714669 entries, 0 to 714668
Data columns (total 8 columns):
 #   Column           Non-Null Count   Dtype   
---  ------           --------------   -----   
 0   order_id         714669 non-null  int32   
 1   product_id       714669 non-null  int16   
 2   quantity         714669 non-null  int8    
 3   unit_price       714669 non-null  float64 
 4   discount_amount  714669 non-null  float64 
 5   promo_id         276316 non-null  category
 6   promo_id_2       206 non-null     category
 7   item_id          714669 non-null  int32   
dtypes: category(2), float64(2), int16(1), int32(2), int8(1)
memory usage: 19.8 MB


In [11]:
# Sắp xếp theo order_id và product_id rồi reset index
df = df.sort_values(["order_id", "product_id"]).reset_index(drop=True)
# Đánh lại item_id theo thứ tự sau khi sắp xếp
df["item_id"] = range(1, len(df) + 1)

# Tổng kết trước khi xuất: không còn trùng, kiểm tra null
print("Tổng số dòng                  :", len(df))
print("Tổng số dòng trùng item_id    :", df.duplicated(subset=["item_id"]).sum())
print("Tổng số null (ngoại trừ promo):", df.drop(columns=["promo_id", "promo_id_2"]).isna().sum().sum())
print("Tổng số null promo_id         :", df["promo_id"].isna().sum())

Tổng số dòng                  : 714669
Tổng số dòng trùng item_id    : 0
Tổng số null (ngoại trừ promo): 0
Tổng số null promo_id         : 438353


In [12]:
# Xuất dữ liệu đã được làm sạch ra file CSV mới
df.to_csv("../SilverData/order_items.csv", index=False)
print("Exported order_items.csv successfully!")

Exported order_items.csv successfully!
